# Дообучение одной папки моделей: MAPLES + IDRiD

Один запуск = три чекпоинта → три эксперимента. Включите **GPU T4** и **Internet**. Подключите оба датасета и папку исходных моделей через Kaggle Inputs.

Перед запуском новые скрипты должны быть опубликованы в GitHub-ветке, указанной ниже. Полный код находится в модулях проекта; существующие скрипты не меняются.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/tvsmsa/segmentation_biomarcers_pp.git'
GIT_REF = 'finetuning'  # ветка с добавленными скриптами
REPO = Path('/kaggle/working/segmentation_biomarcers_pp')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', GIT_REF, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
assert (REPO / 'ml/biomarcers/finetune_experiments/train.py').is_file(), 'Опубликуйте новые скрипты в GIT_REF'


In [ ]:
%pip install segmentation-models-pytorch


## Настройки

Выберите одну из шести папок. Пути `None` ищутся автоматически; при неоднозначности задайте реальные пути из Inputs. `RESUME_FROM` нужен только для восстановления outputs из предыдущей Kaggle-сессии; это папка с `run.json`.


In [ ]:
from ml.biomarcers.finetune_experiments.kaggle import find_dataset, find_group, restore_saved_run, run_cli
from ml.biomarcers.finetune_experiments.models import discover_group

GROUP = 'Deeplab dice'
MAPLES_ROOT = None
IDRID_ROOT = None
MODELS_DIR = None
RESUME_FROM = None
EPOCHS, PATIENCE = 30, 5
BATCH_SIZE, ACCUMULATION, WORKERS = 2, 8, 2
ENCODER_LR, HEAD_LR, TRANSUNET_LR = 1e-5, 5e-4, 1e-4
MAX_HOURS, SEED, VALID_FRACTION = 10, 42, 0.30
SMOKE_PATCHES = 0  # 2 для короткой проверки, тогда EPOCHS=1
PREPARED = Path('/kaggle/working/prepared')
OUTPUT = Path('/kaggle/working/finetune_results')

MAPLES_ROOT = find_dataset('test_patches.csv', MAPLES_ROOT)
IDRID_ROOT = find_dataset('idrid_dataset.csv', IDRID_ROOT)
MODELS_DIR = find_group(GROUP, MODELS_DIR)
architecture, loss_kind, mapping = discover_group(MODELS_DIR)
RUN = OUTPUT / f"{architecture}_{loss_kind}{'_smoke' if SMOKE_PATCHES else ''}"
restore_saved_run(RESUME_FROM, OUTPUT, RUN.name)
print('Inputs:', MAPLES_ROOT, IDRID_ROOT, MODELS_DIR, '\nOutput:', RUN)
print('Чекпоинты → эксперименты:', {n: p.name for n, p in mapping.items()})


## Подготовка train/valid

70/30 по исходным снимкам: **патчи одного снимка никогда не попадают одновременно в train и valid**. Маски и изображения остаются в Inputs. Создаются новые CSV с Linux-путями и контрольными суммами.


In [ ]:
run_cli('prepare', '--maples-root', MAPLES_ROOT, '--idrid-root', IDRID_ROOT,
        '--output-dir', PREPARED, '--valid-fraction', VALID_FRACTION, '--seed', SEED)


## Дообучение и сохранение

Эксперимент 1: train MAPLES, valid MAPLES. Эксперимент 2: train IDRiD, valid IDRiD. Эксперимент 3: объединённые train и valid.

Сохраняются `best.pth`, `last.pth`, история и конфигурация. `--resume` продолжает незавершённый запуск и пропускает завершённые эксперименты. При исчерпании бюджета незавершённая эпоха повторится в следующем запуске. Для новой сессии сохраните outputs и подключите их через `RESUME_FROM`.


In [ ]:
run_cli('train', '--prepared-dir', PREPARED, '--models-dir', MODELS_DIR,
        '--output-dir', OUTPUT, '--epochs', EPOCHS, '--patience', PATIENCE,
        '--batch-size', BATCH_SIZE, '--accumulation', ACCUMULATION, '--num-workers', WORKERS,
        '--encoder-lr', ENCODER_LR, '--head-lr', HEAD_LR, '--transunet-lr', TRANSUNET_LR,
        '--max-hours', MAX_HOURS, '--seed', SEED, '--smoke-patches', SMOKE_PATCHES, '--resume')


## Тест лучших моделей

Оценка отдельно по датасетам и на их объединении: (1) MAPLES valid + весь IDRiD; (2) IDRiD valid + весь MAPLES; (3) MAPLES valid + IDRiD valid. Все 14 биомаркеров учитываются; предсказание без соответствующей метки считается FP. Valid уже участвовал в выборе модели — это не независимый test.


In [ ]:
run_cli('evaluate', '--prepared-dir', PREPARED, '--run-dir', RUN,
        '--batch-size', BATCH_SIZE, '--num-workers', WORKERS)


## Результаты

Скачайте выходные файлы или сохраните версию ноутбука с outputs. `last.pth` нужен для продолжения, `best.pth` — для тестирования и дальнейшего использования. Сохранность `/kaggle/working` между сессиями не предполагается.


In [ ]:
import pandas as pd
from IPython.display import display, FileLink

summary = RUN / 'results/summary.csv'
display(pd.read_csv(RUN / 'results/progress.csv'))
if summary.exists():
    display(pd.read_csv(summary).query("class_name == 'MEAN'"))
    display(FileLink(str(summary)))
for checkpoint in sorted(RUN.glob('experiment_*/*.pth')):
    display(FileLink(str(checkpoint)))
